IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import os as os
import tqdm as tqdm



PREPROCESSING

In [9]:
t1_train_dir = r'Data\task1_data\images\train'
t1_test_dir = r'Data\task1_data\images\test'

if not os.path.exists(t1_train_dir):
    raise FileNotFoundError(f"Training directory not found: {t1_train_dir}")

if not os.path.exists(t1_test_dir):
    raise FileNotFoundError(f"Test directory not found: {t1_test_dir}")




SAVE PREPROCESSED IMAGES

In [ ]:
# ResNet requires 224x224 inputs.
# Mean and std values are from ImageNet (the dataset ResNet was trained on).
# Normalising to the same distribution ResNet expects ensures meaningful feature extraction.
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def image_to_resnet(image_dir, output_path, has_labels=True):
    tensors = []
    labels = []
    paths = []

    if has_labels:
        # subfolder names are the class labels
        class_names = sorted(os.listdir(image_dir))
        class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}

        # count total images for progress display
        total = sum(
            len(os.listdir(os.path.join(image_dir, c)))
            for c in class_names
            if os.path.isdir(os.path.join(image_dir, c))
        )

        c = 0
        for class_name in class_names:
            class_dir = os.path.join(image_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            for fname in os.listdir(class_dir):
                if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue
                img = Image.open(os.path.join(class_dir, fname)).convert("RGB")
                tensors.append(transform(img))
                labels.append(class_to_idx[class_name])
                paths.append(os.path.join(class_dir, fname))
                c += 1
                print(f'{c}/{total}', end='\r')

        torch.save({
            'tensors': torch.stack(tensors),
            'labels': torch.tensor(labels),
            'paths': paths,
            'class_to_idx': class_to_idx
        }, output_path)

    else:
        # test directory is flat = no subfolders, no labels
        fnames = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        total = len(fnames)
        for c, fname in enumerate(fnames, 1):
            img_path = os.path.join(image_dir, fname)
            img = Image.open(img_path).convert("RGB")
            tensors.append(transform(img))
            paths.append(img_path)
            print(f'{c}/{total}', end='\r')

        torch.save({
            'tensors': torch.stack(tensors),
            'paths': paths
        }, output_path)

    print(f"\nSaved {len(tensors)} images to {output_path}")

image_to_resnet(t1_train_dir, r'Data\task1_data\t1_train_resnet.pt', has_labels=True)
image_to_resnet(t1_test_dir,  r'Data\task1_data\t1_test_resnet.pt',  has_labels=False)


3750/3750
Saved 3750 images to Data\task1_data\t1_train_resnet.pt
1250/1250
Saved 1250 images to Data\task1_data\t1_test_resnet.pt


RESNET FEATURE EXTRACTION

In [ ]:
# load ResNet50, strip the final classification layer to get a 2048-d feature extractor
resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
resnet = nn.Sequential(*list(resnet.children())[:-1])
resnet.eval()

def extract_resnet_features(pt_path, output_path):
    data = torch.load(pt_path)
    tensors = data['tensors']   # shape: (N, 3, 224, 224)

    features = []
    total = len(tensors)

    with torch.no_grad():
        for i, img_tensor in enumerate(tensors, 1):
            feat = resnet(img_tensor.unsqueeze(0))  # (1, 2048, 1, 1)
            feat = feat.squeeze().numpy()           # (2048,)
            features.append(feat)
            print(f'{i}/{total}', end='\r')

    features = np.array(features)  # (N, 2048)

    save_data = {**data, 'features': features}
    torch.save(save_data, output_path)
    print(f"\nExtracted features shape: {features.shape} -> saved to {output_path}")

extract_resnet_features(r'Data\task1_data\t1_train.pt', r'Data\task1_data\t1_train_resnet.pt')
extract_resnet_features(r'Data\task1_data\t1_test.pt',  r'Data\task1_data\t1_test_resnet.pt')
